## 01 · 调用大模型

先把 RAG 的最后一步单独跑通：使用 Python 发送消息并取得大模型的文本答案。

这一节重点看客户端配置、请求参数、消息角色与返回结果。


### 创建客户端

`.env` 中的三项配置作用不同：

| 配置 | 作用 | 大白话 |
| --- | --- | --- |
| `LLM_API_KEY` | 身份认证 | 证明谁在调用，不能公开或提交到 Git |
| `LLM_BASE_URL` | API 地址 | 请求要发送到哪家服务 |
| `LLM_MODEL` | 模型名称 | 到达服务后具体调用哪个模型 |

`OpenAI(api_key=..., base_url=...)` 创建一个客户端对象 `client`。客户端会保存认证信息和 API 地址，后续请求不必重复填写。它只是调用服务的工具，不是大模型本身。


In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print(
    f"API 已配置, 配置的模型是{LLM_MODEL}"
    if client
    else "未配置 API：保留本地步骤，调用模型的单元会跳过"
)

API 已配置, 配置的模型是deepseek-v4-flash-0731


## 发送最小请求

核心调用是：

```python
response = client.chat.completions.create(...)
```

可以从左向右读：使用 `client` 客户端，进入对话补全接口 `chat.completions`，调用 `create` 创建一次回答。

| 参数 | 作用 |
| --- | --- |
| `model` | 指定本次使用的模型 |
| `messages` | 按顺序提交消息，每条消息包含 `role` 和 `content` |
| `temperature` | 调整生成时的随机程度，事实问答通常设为 `0` |

API 返回的不只是正文，而是一个包含请求编号、用量和候选答案等信息的响应对象。默认只生成一条答案，因此 `choices` 通常只有一个元素：

```python
response.choices[0].message.content
```

`choices[0]` 表示第一个候选答案，`message.content` 是它的正文。默认情况下不存在 `choices[1]`；如果接口支持并在请求中设置 `n=2`，`choices[1]` 才会是模型针对同一个问题生成的第二个备选答案。它不是对第一条的补充，也不代表一定更差。


In [5]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "用两句话解释什么是向量检索。"}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("请配置 LLM_API_KEY 或 OPENAI_API_KEY")

向量检索是一种通过计算数据向量之间的相似度（如余弦相似度或欧氏距离）来快速查找最相关项的技术。它先将文本、图片等内容映射为高维向量，再在向量空间中搜索与查询向量最接近的邻居。


### 一次生成两条候选答案

如果当前模型服务支持 `n` 参数，可以设置 `n=2`，让模型针对同一个问题一次返回两条备选答案。它不是让模型并发处理两个不同任务，也不保证服务端一定并行计算。

多候选常见于以下场景：

- **创意生成**：一次生成多个标题、广告语或商品描述，供人挑选。
- **程序筛选**：生成多个答案，再由规则、评分模型或人工审核选出一条。
- **投票校验**：复杂问题生成多条推理结果，再根据一致性选择结论。

多生成候选会增加输出量和费用。普通聊天和 RAG 事实问答通常使用 `n=1`；多条答案也不能弥补检索资料本身的错误。

百炼的思考模式要求 `n=1`，因此这个示例通过 `extra_body={"enable_thinking": False}` 关闭思考模式。实际代码不要假定答案数量固定，而应遍历 `response.choices`。


In [22]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "用一句话解释什么是 RAG。"}],
        temperature=1,
        n=2,
        extra_body={"enable_thinking": False},
    )
    for index, choice in enumerate(response.choices, start=1):
        print(f"候选答案 {index}：{choice.message.content}")
else:
    print("请配置 LLM_API_KEY 或 OPENAI_API_KEY")


候选答案 1：RAG（检索增强生成）是一种让大模型在回答问题前，先从外部知识库中检索相关信息，再将检索结果作为参考来生成答案的技术，以弥补模型自身知识的不足。
候选答案 2：RAG（检索增强生成）是一种让大语言模型在回答前，先从外部知识库中检索相关信息作为参考，从而生成更准确、更符合事实的回答的技术方法。


## system 和 user

`system` 规定模型的回答方式，`user` 提出具体问题。RAG 的“只能根据资料回答”通常放在 system 消息里。


In [3]:
messages = [
    {"role": "system", "content": "你是服饰箱包知识库助手，只根据用户提供的资料回答。"},
    {"role": "user", "content": "SKU-JK902 是什么产品？"},
]

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL, messages=messages, temperature=0
    )
    print(response.choices[0].message.content)
else:
    print(messages)

SKU-JK902 没有在您提供的资料中提及，我无法确认它是什么产品。请提供相关产品信息，我再为您查询。


## temperature

`temperature` 调整候选词的概率分布，但不保证某一次输出必然不同。用较长的开放式续写放大差异，再比较同组文本的平均相似度。


In [ ]:
from difflib import SequenceMatcher
from itertools import combinations

def average_similarity(answers):
    pairs = combinations(answers, 2)
    scores = [SequenceMatcher(None, left, right).ratio() for left, right in pairs]
    return sum(scores) / len(scores)

if client:
    prompt = (
        "续写一个60到80字的微型故事：凌晨三点，办公室的打印机突然吐出一张来自明天的通知。"
        "故事必须有完整结局，只返回故事正文。"
    )
    for temperature in [0, 1]:
        answers = []
        for _ in range(3):
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            answers.append(response.choices[0].message.content.strip())
        print(f"temperature={temperature}")
        for index, answer in enumerate(answers, start=1):
            print(f"{index}. {answer}")
        print(f"组内平均相似度：{average_similarity(answers):.2f}\n")
else:
    print("未调用模型：配置 API 后可比较 temperature=0 和 1 的输出差异")
